# Cleaning 2.2 - Clean unique values in panel dataset

This notebook does the following:
    (1) Cleans unique values of variables using the clean_unique_values function
    (2) Drops all rows to be excluded due to unused equipment or different types of equipment
    (3) Cleans affected variables using the clean_affected_vars function
    (5) Cleans the screens variable when total no given instead of per unit no
    (6) Splits types where needed
    (7) Deals with special cases

## Set-up

In [1]:
# Indicator for new variables to be cleaned (default is False, set to True if decide to clean more variables)
new_vars_to_clean = False

In [2]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[0]
sys.path.append(str(CODE_ROOT))
import config
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment
import os
from create_empty_cleaning_sheet import create_empty_cleaning_sheet
from unique_values_cleaning import clean_unique_values
from create_empty_aff_vars_sheet import create_empty_aff_vars_sheet
from affected_vars_cleaning import clean_affected_vars
from split_types_cleaning import clean_split_types, reassign_type_no


In [3]:
# Reload updated split_types_cleaning module after edits
import importlib
import split_types_cleaning
importlib.reload(split_types_cleaning)
from split_types_cleaning import clean_split_types, reassign_type_no


In [4]:
# Load data
equipment = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_1.csv",
    keep_default_na=False,  # Keep "None" as a string, not NaN
    na_values=[""] # Only treat empty strings as NaN
)

In [5]:
# Load data dictionaries
equipment_data_dict = pd.read_excel(config.DATA_DICTIONARIES / "data_dictionary.xlsx", sheet_name="Equipment")

In [6]:
# Load equipment to exclude list
exclusion_list = pd.read_excel(config.CLEANING_WORKBOOKS / "equipment_to_exclude.xlsx", sheet_name = "Exclusion_List")

In [7]:
# Load equipment special cases list
equipment_special_cases = pd.read_excel(config.CLEANING_WORKBOOKS / "equipment_special_cases.xlsx", sheet_name = "Special_Cases")

## (1) Clean unique values of variables

In [8]:
# Create or update the excel workbook with empty sheets
if new_vars_to_clean:

    # File name
    file_name = config.CLEANING_WORKBOOKS / "panel_cleaning_workbook.xlsx"

    # Equipment questions (one sheet per variable)
    for _, row in equipment_data_dict.iterrows():   
        sheet_name = row["Variable"]
        free_text = row["Free text"] == "Y"
        mc_fc_vars = free_text
        create_empty_cleaning_sheet(file_name, sheet_name, 
                                    comment = True, 
                                    free_text = free_text, 
                                    mc_fc_vars = mc_fc_vars)
        
    # Share and EL check variables
    for var in ["share", "el_check"]:
        create_empty_cleaning_sheet(
            file_name, sheet_name=var,
            comment=True, 
            free_text=False,
            mc_fc_vars=False)

In [9]:
# Run cleaning loop. This will: 
# 1. Merge our data to the cleaning workbook
# 2. Create cleaned variables
# 3. produce a report of the cleaning process
# 4. Update the cleaning workbook with all uncleaned values

file_name = config.CLEANING_WORKBOOKS / "panel_cleaning_workbook.xlsx"

# Equipment questions (one sheet per variable)
for _, row in equipment_data_dict.iterrows():
    var_name = row["Variable"]
    sheet_name = var_name
    free_text = row["Free text"] == "Y"
    mc_fc_vars = free_text
    equipment=clean_unique_values(df=equipment, file_name=file_name, 
                                  sheet_name=sheet_name, var_name=var_name, 
                                  comment=True, free_text=free_text, 
                                  mc_fc_vars=mc_fc_vars, 
                                  report=False, 
                                  affected_vars=True)

# Share and EL check variables
for var in ["share", "el_check"]:
    equipment=clean_unique_values(df=equipment, file_name=file_name, 
                                  sheet_name=var, var_name=var, 
                                  comment=True, free_text=False, 
                                  mc_fc_vars=False, report=True,
                                  affected_vars=True)

Cleaning progress for share:
Total unique value combinations: 96
Cleaned combinations: 44
Pending combinations: 51
Excluded combinations: 1
Unchecked combinations: 0
Cleaning progress for el_check:
Total unique value combinations: 59
Cleaned combinations: 33
Pending combinations: 25
Excluded combinations: 1
Unchecked combinations: 0


In [10]:
# For "pending" values, print the labgroupids, survey and variable names and value so that we can check the surveys

# Equipment questions - print labgroupids and survey for pending values
for _, row in equipment_data_dict.iterrows():
    var_name = row["Variable"]
    for s in ["BL", "EL"]:
        for e in ["fc", "fridge", "freezer", "ult", "glassware", "microbio", "cryostat", "bath", "incubator", "heater", "it"]:
            pending_values = equipment[
                (equipment[f"{var_name}_status"] == "Pending") &
                (equipment["survey"] == s) & 
                (equipment["equipment"] == e)
            ]["labgroupid"].tolist()
            if pending_values:
                print(f"Pending values for {var_name}, survey {s}, equipment {e}: {pending_values}")


In [11]:
# For FCs, view all the relevant columns to identify gaps in the data

# List of FC variables (controller_type, sash_width, face_velocity, number, lifted, hours_open, days, surface)
fc_vars = ["controller_type", "sash_width", "face_velocity", "number", "lifted", "hours_open", "days", "surface"]

fc_data = equipment[equipment["equipment"] == "fc"]

# Keep only relevant cols for FC variables
cols_to_keep = ["labgroupid", "survey"] + [f"{var}_clean" for var in fc_vars]
fc_data = fc_data[cols_to_keep]

In [12]:
# For ULTs, view all the relevant columns to identify gaps in the data
ult_data = equipment[equipment["equipment"] == "ult"]
# Keep only relevant cols for ULT variables (ult_type, size_ult, temp_ult, number, seals, spacing, filter, door_openings)
ult_vars = ["ult_type", "size_ult", "temp_ult", "number", "seals", "spacing", "filter", "door_openings"]
cols_to_keep_ult = ["labgroupid", "survey"] + [f"{var}_clean" for var in ult_vars]
ult_data = ult_data[cols_to_keep_ult]

## (2) Drop all rows to be excluded

In [13]:
# If any *_status column is "Exclude", mark the row for exclusion
status_cols = [c for c in equipment.columns if c.endswith("_status")]

equipment["exclude_status"] = (
    equipment[status_cols].eq("Exclude").any(axis=1) if status_cols else False
)

# For each equipment, print the number of rows with "exclude" = True
for eq in equipment["equipment"].unique():
    num_exclude = equipment[(equipment["equipment"] == eq) & (equipment["exclude_status"] == True)].shape[0]
    print(f"Number of rows to exclude based on cleaning workbook for {eq}: {num_exclude}")

# Now count the number of rows to exclude based on the exclusion list

exclusion_list["exclude_list"] = True  # Add an "exclude" column to the exclusion list

# Merge the exclusion list with the equipment data
equipment = equipment.merge(exclusion_list[["labgroupid", "equipment", "type_no", "survey", "exclude_list"]
                                        ], on=["labgroupid", "equipment", "type_no", "survey"], how="left")

# If "exclude" is True in either the cleaning workbook or the exclusion list, mark the row for exclusion
equipment["exclude"] = equipment["exclude_status"] | equipment["exclude_list"]
equipment.drop(columns=["exclude_status", "exclude_list"], inplace=True)

# For each equipment, print the number of rows with "exclude" = True after merging with exclusion list
for eq in equipment["equipment"].unique():
    num_exclude = equipment[(equipment["equipment"] == eq) & (equipment["exclude"] == True)].shape[0]
    print(f"Number of rows to exclude based on cleaning workbook and exclusion list for {eq}: {num_exclude}")

# Drop rows marked for exclusion (and exclude variable)
equipment = equipment[~equipment["exclude"]].drop(columns=["exclude"]).copy()

Number of rows to exclude based on cleaning workbook for fc: 6
Number of rows to exclude based on cleaning workbook for fridge: 7
Number of rows to exclude based on cleaning workbook for freezer: 2
Number of rows to exclude based on cleaning workbook for ult: 16
Number of rows to exclude based on cleaning workbook for glassware: 21
Number of rows to exclude based on cleaning workbook for microbio: 10
Number of rows to exclude based on cleaning workbook for bath: 14
Number of rows to exclude based on cleaning workbook for incubator: 5
Number of rows to exclude based on cleaning workbook for heater: 14
Number of rows to exclude based on cleaning workbook for it: 22
Number of rows to exclude based on cleaning workbook for cryostat: 21
Number of rows to exclude based on cleaning workbook and exclusion list for fc: 10
Number of rows to exclude based on cleaning workbook and exclusion list for fridge: 7
Number of rows to exclude based on cleaning workbook and exclusion list for freezer: 2
Nu

## (3) Deal with affected variables

In [14]:
# Create affected variables cleaning sheet if doesn't already exist
affected_vars_cleaning_file = config.CLEANING_WORKBOOKS / "affected_variables_cleaning.xlsx"

id_cols = ["labgroupid", "equipment", "type_no", "survey"]

create_empty_aff_vars_sheet(
    file_name=affected_vars_cleaning_file,
    sheet_name="Panel Vars",
    id_cols=id_cols
)

Sheet Panel Vars already exists. No changes made.


In [15]:
# Clean affected variables
equipment = clean_affected_vars(
    df=equipment,
    file_name=affected_vars_cleaning_file,
    sheet_name="Panel Vars",
    data_dict = None,
    id_cols=id_cols
)

No new affected variable cases found.
Pulled back cleaned values for 5 affected variable case(s) where value_changed == 'Y'.


## (5) Deal with special cases

In [16]:
# Print the special cases
print("Special cases:")
print(equipment_special_cases["action"].unique())

Special cases:
['Copy all values from type 1 to type 2 except monitor and screen'
 'Duplicate EL to BL (forgotten on first visit) except door_openings'
 'Duplicate EL to BL (counting error in BL)']


In [17]:
# Deal with special cases (currently handled case-by-case)

# Merge special-case actions onto equipment
equipment = equipment.merge(
    equipment_special_cases,
    on=["labgroupid", "equipment", "type_no", "survey"],
    how="left"
)

# Case 1: "Copy all values from type 1 to type 2 except monitor and screen"
# Keep monitor_clean/status from type_no == 2, but copy all other *_clean and *_status values from type_no == 1
action_text = "Copy all values from type 1 to type 2 except monitor and screen"

if "action" in equipment.columns:
    flagged = equipment[
        equipment["action"].astype("string").str.strip() == action_text
    ][["labgroupid", "equipment", "survey"]].drop_duplicates()

    if not flagged.empty:
        copy_cols = [
            c for c in equipment.columns
            if (c.endswith("_clean") or c.endswith("_status"))
            and c not in ("monitor_clean", "screens_clean", "monitor_status", "screens_status")
        ]

        source = equipment[
            equipment["type_no"] == 1
        ][["labgroupid", "equipment", "survey"] + copy_cols].copy()
        source = source.rename(columns={c: f"{c}__src" for c in copy_cols})

        equipment = equipment.merge(
            source,
            on=["labgroupid", "equipment", "survey"],
            how="left"
        )

        flagged_index = flagged.set_index(["labgroupid", "equipment", "survey"]).index
        target_mask = (
            (equipment["type_no"] == 2)
            & equipment.set_index(["labgroupid", "equipment", "survey"]).index.isin(flagged_index)
        )

        for c in copy_cols:
            src_col = f"{c}__src"
            equipment.loc[target_mask, c] = equipment.loc[target_mask, src_col]

        drop_src_cols = [f"{c}__src" for c in copy_cols if f"{c}__src" in equipment.columns]
        if drop_src_cols:
            equipment = equipment.drop(columns=drop_src_cols)

        print(f"Applied special case to {int(target_mask.sum())} type_no==2 row(s).")
    else:
        print("No rows flagged for this exact action text.")
else:
    print(f"Special-case action column not found: 'action'")

# Case 2: Copy all *_clean and *_status values from EL to BL rows
# (e.g. "Duplicate EL to BL (forgotten on first visit) except door_openings")
action_text = "Duplicate EL to BL (forgotten on first visit) except door_openings"

if "action" in equipment.columns:
    flagged = equipment[
        equipment["action"].astype("string").str.strip() == action_text
    ][["labgroupid", "equipment", "type_no"]].drop_duplicates()

    if not flagged.empty:
        copy_cols = [
            c for c in equipment.columns
            if c.endswith("_clean") or c.endswith("_status")
            and c not in ("door_openings_clean", "door_openings_status")
        ]

        source = equipment[
            equipment["survey"] == "EL"
        ][["labgroupid", "equipment", "type_no"] + copy_cols].copy()
        source = source.rename(columns={c: f"{c}__src" for c in copy_cols})

        equipment = equipment.merge(
            source,
            on=["labgroupid", "equipment", "type_no"],
            how="left"
        )

        flagged_index = flagged.set_index(["labgroupid", "equipment", "type_no"]).index
        target_mask = (
            (equipment["survey"] == "BL")
            & equipment.set_index(["labgroupid", "equipment", "type_no"]).index.isin(flagged_index)
        )

        for c in copy_cols:
            src_col = f"{c}__src"
            equipment.loc[target_mask, c] = equipment.loc[target_mask, src_col]

        drop_src_cols = [f"{c}__src" for c in copy_cols if f"{c}__src" in equipment.columns]
        if drop_src_cols:
            equipment = equipment.drop(columns=drop_src_cols)

        print(f"Applied EL-to-BL copy (except door_openings) to {int(target_mask.sum())} BL row(s).")
    else:
        print("No rows flagged for this exact action text.")

# Case 3: Copy all *_clean and *_status values from EL to BL rows
# (e.g. "Duplicate EL to BL (counting error in BL)")
action_texts = ["Duplicate EL to BL (forgotten on first visit)", "Duplicate EL to BL (counting error in BL)"]

if "action" in equipment.columns:
    flagged = equipment[
        equipment["action"].astype("string").str.strip().isin(action_texts)
    ][["labgroupid", "equipment", "type_no"]].drop_duplicates()

    if not flagged.empty:
        copy_cols_el = [
            c for c in equipment.columns
            if c.endswith("_clean") or c.endswith("_status")
        ]

        source = equipment[
            equipment["survey"] == "EL"
        ][["labgroupid", "equipment", "type_no"] + copy_cols_el].copy()
        source = source.rename(columns={c: f"{c}__src" for c in copy_cols_el})

        equipment = equipment.merge(
            source,
            on=["labgroupid", "equipment", "type_no"],
            how="left"
        )

        flagged_index = flagged.set_index(["labgroupid", "equipment", "type_no"]).index
        target_mask = (
            (equipment["survey"] == "BL")
            & equipment.set_index(["labgroupid", "equipment", "type_no"]).index.isin(flagged_index)
        )

        for c in copy_cols_el:
            src_col = f"{c}__src"
            equipment.loc[target_mask, c] = equipment.loc[target_mask, src_col]

        drop_src_cols = [f"{c}__src" for c in copy_cols_el if f"{c}__src" in equipment.columns]
        if drop_src_cols:
            equipment = equipment.drop(columns=drop_src_cols)

        print(f"Applied EL-to-BL copy to {int(target_mask.sum())} BL row(s).")
    else:
        print("No rows flagged for EL-to-BL copy actions.")


Applied special case to 2 type_no==2 row(s).
Applied EL-to-BL copy (except door_openings) to 1 BL row(s).
Applied EL-to-BL copy to 2 BL row(s).


## (6) Clean the screens variable

In [18]:
# If screens_status is "Total", create screens_clean_per_unit by dividing screens by number
equipment.loc[equipment["screens_status"] == "Total", 
                      "screens_clean_per_unit"] = equipment["screens_clean"] / equipment["number_clean"]

# Replace screens_clean with screens_clean_per_unit for rows where screens_status is "Total"
equipment.loc[equipment["screens_status"] == "Total",
               "screens_clean"] = equipment.loc[equipment["screens_status"] == "Total", "screens_clean_per_unit"]

# Drop the screens_clean_per_unit column as it's no longer needed
equipment.drop(columns=["screens_clean_per_unit"], inplace=True, errors="ignore")

## (7) Split types where needed

In [19]:
# Export/import split-type rows via cleaning workbook
split_types_cleaning_file = config.CLEANING_WORKBOOKS / "split_types_cleaning.xlsx"

split_id_cols = ["labgroupid", "equipment", "type_no", "survey"]
split_clean_cols = [c for c in equipment.columns if c.endswith("_clean")]

equipment = clean_split_types(
    df=equipment,
    file_name=split_types_cleaning_file,
    sheet_name="Panel Split Types",
    id_cols=split_id_cols,
    clean_cols=split_clean_cols,
    split_status="Split",
    auto_assign_type_no=True,
    split_type_start=11,
)


No new Original rows to append.
Replaced 43 original row(s) with 94 split row(s).
Split-types created per original row (split_count -> num_originals):
  2 -> 39
  3 -> 2
  5 -> 2
Total original rows replaced: 43


/Users/drutna/Library/CloudStorage/OneDrive-UniversitätZürichUZH/lab-experiment/1_Cleaning/split_types_cleaning.py:207: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([out_remaining, new_rows_df[out.columns.tolist() + [c for c in new_rows_df.columns if c not in out.columns]]], ignore_index=True)


In [20]:
# Compare BL and EL split rows and save the result workbook
split_rows_differences_file = config.CLEANING_WORKBOOKS / "split_rows_differences_bl_el.xlsx"
split_types = pd.read_excel(split_types_cleaning_file, sheet_name="Panel Split Types", dtype=str)
split_types = split_types[split_types["split_type"].astype(str).str.strip().str.lower().eq("split")].copy()
bl = split_types[split_types["survey"].eq("BL")].copy()
el = split_types[split_types["survey"].eq("EL")].copy()
print(f"Split rows in workbook: total={len(split_types)}, BL={len(bl)}, EL={len(el)}")

group_cols = ["labgroupid", "equipment"]
sig_cols = [c for c in split_types.columns if c not in {"split_type", "original_key", "survey", "type_no"}]

def _variants(df):
    return {k if isinstance(k, tuple) else (k,): {tuple(r.get(c, pd.NA) for c in sig_cols) for _, r in g.iterrows()} for k, g in df.groupby(group_cols, dropna=False)} if not df.empty else {}

bl_sets, el_sets = _variants(bl), _variants(el)
summary, mismatch_rows = [], []
identical = different = only_bl = only_el = 0

for key in sorted(set(bl_sets) | set(el_sets)):
    bl_keys = sorted(bl.loc[(bl["labgroupid"] == key[0]) & (bl["equipment"] == key[1]), "original_key"].dropna().astype(str).unique().tolist()) if key in bl_sets else []
    el_keys = sorted(el.loc[(el["labgroupid"] == key[0]) & (el["equipment"] == key[1]), "original_key"].dropna().astype(str).unique().tolist()) if key in el_sets else []
    if key in bl_sets and key in el_sets and bl_sets[key] == el_sets[key]:
        identical += 1
        continue
    status = "different" if key in bl_sets and key in el_sets else ("only_bl" if key in bl_sets else "only_el")
    different += int(status == "different")
    only_bl += int(status == "only_bl")
    only_el += int(status == "only_el")
    mismatch_rows.append({"labgroupid": key[0], "equipment": key[1], "status": status, "bl_original_key": "; ".join(bl_keys), "el_original_key": "; ".join(el_keys)})

summary_df = pd.DataFrame([
    {"metric": "identical_split_sets", "value": identical},
    {"metric": "different_split_sets", "value": different},
    {"metric": "only_bl", "value": only_bl},
    {"metric": "only_el", "value": only_el},
    {"metric": "total_original_groups_compared", "value": len(set(bl_sets) | set(el_sets))},
])
mismatch_df = pd.DataFrame(mismatch_rows).sort_values(group_cols, kind="stable") if mismatch_rows else pd.DataFrame(columns=["labgroupid", "equipment", "status", "bl_original_key", "el_original_key"])

with pd.ExcelWriter(split_rows_differences_file, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="summary", index=False)
    mismatch_df.to_excel(writer, sheet_name="differences", index=False)

print("BL vs EL split-variants comparison:")
print(f"  Identical split sets: {identical}")
print(f"  Different split sets: {different}")
print(f"  Only BL present: {only_bl}")
print(f"  Only EL present: {only_el}")

Split rows in workbook: total=94, BL=52, EL=42
BL vs EL split-variants comparison:
  Identical split sets: 15
  Different split sets: 2
  Only BL present: 4
  Only EL present: 0


## Clean up and save processed data

In [21]:
# Final renumbering of type_no after exclusions, split handling, and special cases
equipment = reassign_type_no(
    equipment,
    group_cols=("labgroupid", "equipment", "survey"),
    type_col="type_no",
    start=1,
)

In [22]:
# For the cleaned variables, keep only the cleaned version and drop all original variables and variables used in cleaning
clean_cols = [col for col in equipment.columns if col.endswith("_clean")]
for col in clean_cols:
    original_col = col.replace("_clean", "")
    aff_vars_col = col.replace("_clean", "_aff_vars")
    status_col = col.replace("_clean", "_status")
    # Also _co and _fc if they exist
    co_col = col.replace("_clean", "_co")
    fc_col = col.replace("_clean", "_fc")
    fc_co_col = col.replace("_clean", "_fc_co")
    cols_to_drop = [original_col, aff_vars_col, status_col, co_col, fc_col, fc_co_col]
    equipment.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    # Rename cleaned column to original name
    equipment.rename(columns={col: original_col}, inplace=True)

# Drop "action" and "original_key" cols
equipment.drop(columns=["action", "original_key"], inplace=True, errors="ignore")

In [23]:
# Drop if missing everything apart from labgroupid, survey, equipment, type_no
keep_cols = ["labgroupid", "survey", "equipment", "type_no"]
cols_to_check = equipment.columns.difference(keep_cols)
missing_everything = equipment[cols_to_check].isna() | (equipment[cols_to_check] == "")
print(f"Number of rows missing everything apart from {keep_cols}: {missing_everything.all(axis=1).sum()}")
equipment = equipment[~missing_everything.all(axis=1)].copy()

Number of rows missing everything apart from ['labgroupid', 'survey', 'equipment', 'type_no']: 0


In [24]:
# Save processed data
equipment.to_csv(config.PROCESSED_DATA / "panel_processed_2.csv", index =False)